# Pipeline GDELT - Traitement des données

Objectif :
- Collecter les données GDELT
- Nettoyer et transformer
- Enrichir les données
- Sauvegarder en format exploitable

In [ ]:
### Cellule pour l'importation des biliotheques
import pandas as pd          # Importer pandas pour manipuler les données en tableaux (DataFrames)
import numpy as np           # Importer numpy pour les calculs numériques et gestion des valeurs manquantes
from tqdm import tqdm        # Importer tqdm pour afficher des barres de progression
import os                    # Importer os pour interagir avec le système de fichiers (créer dossiers, vérifier tailles)
from pathlib import Path     # Manipuler les chemins de fichiers et dossiers de manière portable et orientée objet


# Afficher toutes les colonnes sans limites
pd.set_option('display.max_columns', None)   # Afficher toutes les colonnes d’un DataFrame pandas sans limitation
pd.set_option('display.max_rows', 100)       # Afficher jusqu'à 100 lignes

In [ ]:
### Cellule pour le chargement de la dataset brut
CHEMIN_RAW = '../data/raw/benin_raw.csv'  # Chemin vers le fichier brut extrait de BigQuery

df_raw = pd.read_csv(CHEMIN_RAW, low_memory=False) # Lire le fichier CSV brut et le stocker dans un DataFrame

print(f" Lignes : {df_raw.shape[0]}")     # Afficher le nombre de lignes du DataFrame
print(f" Colonnes : {df_raw.shape[1]}")   # Afficher le nombre de colonnes du DataFrame

print(f"\n Colonnes disponibles :")
for i, col in enumerate(df_raw.columns.tolist(), 1):
    print(f"  {i:02d}. {col}")              # Afficher la liste de toutes les colonnes disponibles avec leurs rangs


 Lignes : 10722
 Colonnes : 31

 Colonnes disponibles :
  01. GLOBALEVENTID
  02. SQLDATE
  03. MONTHYEAR
  04. YEAR
  05. FractionDate
  06. IsRootEvent
  07. ActionGeo_CountryCode
  08. ActionGeo_FullName
  09. ActionGeo_ADM1Code
  10. ActionGeo_Lat
  11. ActionGeo_Long
  12. Actor1Geo_CountryCode
  13. Actor2Geo_CountryCode
  14. Actor1CountryCode
  15. Actor2CountryCode
  16. Actor1Name
  17. Actor2Name
  18. Actor1Type1Code
  19. Actor2Type1Code
  20. Actor1KnownGroupCode
  21. Actor2KnownGroupCode
  22. EventRootCode
  23. EventBaseCode
  24. EventCode
  25. QuadClass
  26. GoldsteinScale
  27. NumMentions
  28. NumSources
  29. NumArticles
  30. AvgTone
  31. SOURCEURL


In [ ]:
### Cellule pour le diagnostic de la qualite de la dataset
print("=== VALEURS MANQUANTES ===")

missing = df_raw.isnull().sum()             # Compter le nombre de valeurs manquantes (NaN) par colonne

missing_pct = (missing / len(df_raw) * 100).round(2)   # Calculer le pourcentage de valeurs manquantes par colonne, arrondi à 2 décimales

rapport_qualite = pd.DataFrame({            
    'manquants': missing,                   # Colonne avec le nombre de valeurs manquantes
    'pourcentage': missing_pct              # Colonne avec le pourcentage de valeurs manquantes
}).sort_values('pourcentage', ascending=False)  # Créer un DataFrame récapitulatif trié du plus au moins de valeurs manquantes

print(rapport_qualite[rapport_qualite['manquants'] > 0])  # Afficher uniquement les colonnes qui ont au moins une valeur manquante

print("\n=== TYPES DE COLONNES ===")
print(df_raw.dtypes)        # Afficher le type de données de chaque colonne (int, float, object...)

print("\n=== APERÇU ===")
df_raw.head(3)              # Afficher les 3 premières lignes pour vérifier visuellement les données

=== VALEURS MANQUANTES ===
                       manquants  pourcentage
Actor2KnownGroupCode       10630        99.14
Actor1KnownGroupCode       10571        98.59
Actor2Type1Code             7705        71.86
Actor1Type1Code             7515        70.09
Actor2CountryCode           4145        38.66
Actor1CountryCode           2775        25.88
Actor2Geo_CountryCode       2057        19.18
Actor2Name                  2057        19.18
Actor1Geo_CountryCode        748         6.98
Actor1Name                   748         6.98

=== TYPES DE COLONNES ===
GLOBALEVENTID              int64
SQLDATE                    int64
MONTHYEAR                  int64
YEAR                       int64
FractionDate             float64
IsRootEvent                int64
ActionGeo_CountryCode        str
ActionGeo_FullName           str
ActionGeo_ADM1Code           str
ActionGeo_Lat            float64
ActionGeo_Long           float64
Actor1Geo_CountryCode        str
Actor2Geo_CountryCode        str
Actor1Count

,GLOBALEVENTID,SQLDATE,MONTHYEAR,YEAR,FractionDate,IsRootEvent,ActionGeo_CountryCode,ActionGeo_FullName,ActionGeo_ADM1Code,ActionGeo_Lat,ActionGeo_Long,Actor1Geo_CountryCode,Actor2Geo_CountryCode,Actor1CountryCode,Actor2CountryCode,Actor1Name,Actor2Name,Actor1Type1Code,Actor2Type1Code,Actor1KnownGroupCode,Actor2KnownGroupCode,EventRootCode,EventBaseCode,EventCode,QuadClass,GoldsteinScale,NumMentions,NumSources,NumArticles,AvgTone,SOURCEURL
0,1281664740,20251231,202512,2025,2025.989,1,BN,Benin,BN,9.5,2.25,BN,NaN,BEN,NaN,BENIN,NaN,NaN,NaN,NaN,NaN,4,43,43,1,2.8,10,1,10,-1.09589,https://www.rewmi.com/koulibaly-la-rdc-va-fair...
1,1281664663,20251231,202512,2025,2025.989,1,BN,Benin,BN,9.5,2.25,NaN,BN,NaN,BEN,NaN,BENIN,NaN,NaN,NaN,NaN,4,42,42,1,1.9,10,1,10,-1.09589,https://www.rewmi.com/koulibaly-la-rdc-va-fair...
2,1281698938,20251231,202512,2025,2025.989,1,BN,Benin,BN,9.5,2.25,BN,BN,BEN,BEN,BENIN,BENIN,NaN,MIL,NaN,NaN,19,190,190,4,-10.0,2,1,2,-9.76801,https://www.firstpost.com/world/2025-year-ende...


In [ ]:
### Cellule pour le nettoyage de la dataset
df = df_raw.copy()    # Copier le DataFrame brut pour ne jamais modifier l'original

# ── 1. CORRECTION DES TYPES ───────────────────────────────

df['SQLDATE'] = pd.to_datetime(     # Convertir SQLDATE en datetime Python
    df['SQLDATE'].astype(str),
    format='%Y%m%d',                # format='%Y%m%d' : la date est au format YYYYMMDD (ex: 20250427)
    errors='coerce'                 # errors='coerce' : les dates invalides deviennent NaT au lieu de planter
)


df['FractionDate'] = pd.to_numeric(df['FractionDate'], errors='coerce')
# Convertir la date décimale en nombre (ex: 2025.322)

df['GoldsteinScale'] = pd.to_numeric(df['GoldsteinScale'], errors='coerce')
# Convertir le score de stabilité en décimal

df['AvgTone'] = pd.to_numeric(df['AvgTone'], errors='coerce')
# Convertir le ton médiatique en décimal

df['ActionGeo_Lat'] = pd.to_numeric(df['ActionGeo_Lat'], errors='coerce')
# Convertir la latitude en décimal

df['ActionGeo_Long'] = pd.to_numeric(df['ActionGeo_Long'], errors='coerce')
# Convertir la longitude en décimal

df['NumMentions'] = pd.to_numeric(df['NumMentions'], errors='coerce')
# Convertir le nombre de mentions en entier

df['NumSources'] = pd.to_numeric(df['NumSources'], errors='coerce')
# Convertir le nombre de sources en entier

df['NumArticles'] = pd.to_numeric(df['NumArticles'], errors='coerce')
# Convertir le nombre d'articles en entier

df['IsRootEvent'] = pd.to_numeric(df['IsRootEvent'], errors='coerce')
# Convertir IsRootEvent en entier (0 ou 1)

df['QuadClass'] = pd.to_numeric(df['QuadClass'], errors='coerce')
# Convertir QuadClass en entier (1, 2, 3 ou 4)

print(" Types corrigés")

# ── 2. SUPPRESSION DES DOUBLONS ───────────────────────────

nb_avant = len(df)     # Mémoriser le nombre de lignes avant suppression

df = df.drop_duplicates(subset=['GLOBALEVENTID'])       # Supprimer les lignes en double en se basant sur l'identifiant unique

print(f" Doublons supprimés : {nb_avant - len(df):,}")

# ── 3. SUPPRESSION DES DATES INVALIDES ────────────────────

nb_avant = len(df)
df = df.dropna(subset=['SQLDATE'])   # Supprimer les lignes dont la date est invalide ou manquante

print(f" Lignes sans date supprimées : {nb_avant - len(df):,}")

print(f"\n Dataset après nettoyage : {df.shape[0]:,} lignes x {df.shape[1]} colonnes")
print(f" Période couverte : {df['SQLDATE'].min().date()} → {df['SQLDATE'].max().date()}")

 Types corrigés
 Doublons supprimés : 0
 Lignes sans date supprimées : 0

 Dataset après nettoyage : 10,722 lignes x 31 colonnes
 Période couverte : 2025-01-01 → 2025-12-31


In [ ]:
# Cellule pour afficher toutes les valeurs uniques de ActionGeo_FullName afin de faire la repartition par zone
valeurs_uniques = df['ActionGeo_FullName'].unique()  # Extraire toutes les valeurs uniques de la colonne 'ActionGeo_FullName'

print(f"Nombre de valeurs uniques : {len(valeurs_uniques)}")  # Afficher le nombre total de valeurs uniques

print("\nListe complète :")  # Afficher un titre pour la liste

for i, valeur in enumerate(sorted(valeurs_uniques, key=lambda x: str(x)), 1):  
    # Parcourir les valeurs triées (en convertissant en string pour éviter les erreurs avec NaN)

    print(f"  {i:03d}. {valeur}")  
    # Afficher chaque valeur avec un numéro formaté sur 3 chiffres

Nombre de valeurs uniques : 83

Liste complète :
  001. Abomey, Zou, Benin
  002. Affame, Benin (general), Benin
  003. Ahomadegbe, Benin (general), Benin
  004. Akassato, Benin (general), Benin
  005. Akoko, Atakora, Benin
  006. Akrake, Benin (general), Benin
  007. Alafia, Benin (general), Benin
  008. Alassane, Benin (general), Benin
  009. Alibori, Alibori, Benin
  010. Allada, Benin (general), Benin
  011. Atacora, Atakora, Benin
  012. Atakora, Atakora, Benin
  013. Atlantique, Atlantique, Benin
  014. Avlekete, Atlantique, Benin
  015. Avlo, Benin (general), Benin
  016. Banikoara, Alibori, Benin
  017. Bariba, Benin (general), Benin
  018. Bembereke, Borgou, Benin
  019. Benin
  020. Bopa, Benin (general), Benin
  021. Borgou, Borgou, Benin
  022. Bouca, Borgou, Benin
  023. Cadjehoun, Benin (general), Benin
  024. Cococodji, Benin (general), Benin
  025. Collines, Collines, Benin
  026. Cotonou, Littoral, Benin
  027. Couffo, Kouffo, Benin
  028. Dangbo, Benin (general), Beni

In [ ]:
### Cellule pour l'enrichissement de la dataset
# ── COLONNES TEMPORELLES ──────────────────────────────────

df['mois'] = df['SQLDATE'].dt.month
# Extraire le numéro du mois (1 à 12)

df['trimestre'] = df['SQLDATE'].dt.quarter
# Extraire le trimestre (1 à 4)

df['annee'] = df['SQLDATE'].dt.year
# Extraire l'année

df['mois_annee'] = df['SQLDATE'].dt.to_period('M').astype(str)
# Créer une colonne "YYYY-MM" (ex: "2025-03") pour les graphiques temporels

df['jour_semaine'] = df['SQLDATE'].dt.dayofweek
# Extraire le jour de la semaine (0=Lundi, 6=Dimanche)
# Utile pour détecter des patterns hebdomadaires (Q4)

# ── CATÉGORISATION DU TON MÉDIATIQUE (Q1, Q2, Q3) ────────

def categoriser_ton(tone):
    # Catégoriser le ton médiatique en 4 niveaux
    if pd.isna(tone):
        return 'inconnu'       # Valeur manquante
    elif tone > 3:
        return 'tres_positif'  # Couverture très favorable
    elif tone > 1:
        return 'positif'       # Couverture favorable
    elif tone < -3:
        return 'tres_negatif'  # Couverture très défavorable
    elif tone < -1:
        return 'negatif'       # Couverture défavorable
    else:
        return 'neutre'        # Couverture neutre

df['ton_categorie'] = df['AvgTone'].apply(categoriser_ton)
# Appliquer la fonction à chaque ligne de AvgTone

# ── CATÉGORISATION GOLDSTEIN (Q1, Q3, Q4) ────────────────

def categoriser_goldstein(score):
    # Catégoriser le score de stabilité
    if pd.isna(score):
        return 'inconnu'
    elif score >= 5:
        return 'tres_cooperatif'   # Événement très positif pour la stabilité
    elif score > 0:
        return 'cooperatif'        # Événement positif
    elif score == 0:
        return 'neutre'            # Événement neutre
    elif score >= -5:
        return 'conflictuel'       # Événement négatif
    else:
        return 'tres_conflictuel'  # Événement très déstabilisateur

df['goldstein_categorie'] = df['GoldsteinScale'].apply(categoriser_goldstein)
# Appliquer la fonction à chaque ligne de GoldsteinScale

# ── CATÉGORISATION QUADCLASS (Q2, Q5) ─────────────────────

quadclass_labels = {
    1: 'cooperation_verbale',    # Déclarations de soutien, promesses
    2: 'cooperation_materielle', # Aide concrète, accords signés
    3: 'conflit_verbal',         # Accusations, critiques, menaces
    4: 'conflit_materiel'        # Violence, attaques, arrestations
}
df['quadclass_label'] = df['QuadClass'].map(quadclass_labels)
# Remplacer les codes 1/2/3/4 par des labels lisibles

# ── ZONE GÉOGRAPHIQUE NORD/SUD (Q3) ──────────────────────

communesdepart_nord = [# Alibori
    "Banikoara", "Gogounou", "Kandi", "Karimama", "Malanville", "Segbana", "Alassane", "Gbeke", "Alibori", "Kantoro", "Mehrou", "Toura"

    # Atacora
    "Boukoumbé", "Cobly", "Kerou", "Kouandé", "Matéri", "Natitingou", "Péhunco", "Tanguieta", "Toucountouna", "Akoko", "Kayode", "Atakora", "Porga", "Taiakou", "Tanougou", "Tobre"

    # Donga
    "Bassila", "Copargo", "Djougou", "Ouaké", "Donga"

    # Borgou
    "Bembereke", "Kalale", "Ndali", "Nikki", "Parakou", "Pèrèrè", "Sinendé", "Tchaourou", "Babariba", "Bouca", "Gokana", "Gourou", "Kika" "Borgou", "Sekere"]

communesdepart_centre = [
    # Zou
    "Abomey", "Agbangnizoun", "Bohicon", "Cové", "Djidja",
    "Ouinhi", "Kpota", "Zagnanado", "Zogbodomey", "Dosso", "Zou"

    # Collines
    "Bantè", "Dassa-Zoume", "Glazoue", "Ouèssè", "Savalou", "Savè", "Alafia", "Kokou", "Collines", "Ogou", "Okio"
]



def classifier_zone(nom_lieu):
    # Classifier un lieu béninois en zone nord, centre ou sud
    if pd.isna(nom_lieu):
        return 'inconnu'           # Valeur manquante

    nom_lieu_str = str(nom_lieu).lower()
    # Convertir en minuscules pour ignorer la casse

    # ── Vérifier le nord en premier ──────────────────────
    if any(ville.lower() in nom_lieu_str for ville in communesdepart_nord):
        # Ex : "Kandi, Alibori, Benin" contient "kandi" → nord
        return 'nord'

    # ── Vérifier le centre ensuite ────────────────────────
    elif any(ville.lower() in nom_lieu_str for ville in communesdepart_centre):
        # Ex : "Bohicon, Zou, Benin" contient "bohicon" → centre
        return 'centre'

    # ── Tout le reste = sud ───────────────────────────────
    elif pd.notna(nom_lieu):
        # Ex : "Cotonou, Littoral, Benin" → sud
        return 'sud'

    else:
        return 'inconnu'

df['zone_benin'] = df['ActionGeo_FullName'].apply(classifier_zone)
# Appliquer la classification à chaque ligne

# ── Vérification ─────────────────────────────────────────
print(" Répartition des zones :")
print(df['zone_benin'].value_counts())

# ── EXTRACTION DOMAINE SOURCE (Q1) ────────────────────────

def extraire_domaine(url):
    # Extraire le domaine depuis l'URL pour identifier le média
    if pd.isna(url):
        return 'inconnu'
    try:
        domaine = url.split('/')[2]
        # Prendre la 3ème partie de l'URL (ex: "www.rfi.fr")
        domaine = domaine.replace('www.', '')
        # Supprimer le préfixe "www."
        return domaine
    except:
        return 'inconnu'

df['source_domaine'] = df['SOURCEURL'].apply(extraire_domaine)
# Extraire le domaine de chaque URL source

#Affichge du bilan
print(" Colonnes enrichies ajoutées")
print(f" Dataset enrichi : {df.shape[0]:,} lignes x {df.shape[1]} colonnes")
print(f"\nNouvelles colonnes ajoutées :")
nouvelles = ['mois', 'trimestre', 'annee', 'mois_annee', 'jour_semaine',
             'ton_categorie', 'goldstein_categorie', 'quadclass_label',
             'zone_benin', 'source_domaine']
for col in nouvelles:
    print(f"   {col}")


 Répartition des zones :
zone_benin
sud       10047
nord        523
centre      152
Name: count, dtype: int64
 Colonnes enrichies ajoutées
 Dataset enrichi : 10,722 lignes x 41 colonnes

Nouvelles colonnes ajoutées :
   mois
   trimestre
   annee
   mois_annee
   jour_semaine
   ton_categorie
   goldstein_categorie
   quadclass_label
   zone_benin
   source_domaine


In [ ]:
### Cellule pour le verification des assertions

print(" Vérification des assertions qualité...\n")

# Vérifier que les colonnes critiques sont présentes
colonnes_obligatoires = [
    'GLOBALEVENTID', 'SQLDATE', 'AvgTone',
    'GoldsteinScale', 'ActionGeo_CountryCode'
]
for col in colonnes_obligatoires:
    assert col in df.columns, f" Colonne manquante : {col}"
    # Planter immédiatement si une colonne critique est absente
    print(f"    {col}")

# Vérifier le volume
assert len(df) > 100, f" Trop peu d'événements : {len(df)}"
# S'assurer qu'on a suffisamment de données
print(f"    Volume suffisant : {len(df):,} événements")

# Vérifier l'absence de doublons
assert df['GLOBALEVENTID'].duplicated().sum() == 0, " Doublons détectés !"
print(f"    Aucun doublon")

# Vérifier que toutes les dates sont valides
assert df['SQLDATE'].isna().sum() == 0, " Dates manquantes !"
print(f"    Toutes les dates valides")

# Vérifier la période
assert df['SQLDATE'].min().year >= 2025, " Données avant 2025 détectées !"
print(f"    Période correcte : à partir de 2025")

# Vérifier les latitudes
lat_valides = df['ActionGeo_Lat'].dropna()
if len(lat_valides) > 0:
    assert lat_valides.between(-90, 90).all(), " Latitudes hors limites !"
    print(f"    Latitudes valides")

print("\n Toutes les assertions sont passées !")

 Vérification des assertions qualité...

    GLOBALEVENTID
    SQLDATE
    AvgTone
    GoldsteinScale
    ActionGeo_CountryCode
    Volume suffisant : 10,722 événements
    Aucun doublon
    Toutes les dates valides
    Période correcte : à partir de 2025
    Latitudes valides

 Toutes les assertions sont passées !


In [ ]:
### Cellule de sauvegarde de la dataset 
# Créer le dossier processed s'il n'existe pas
os.makedirs('../data/processed', exist_ok=True)
# exist_ok=True évite une erreur si le dossier existe déjà

# Sauvegarder les données nettoyées de base
df.to_csv('../data/processed/benin_clean.csv', index=False)
# index=False pour ne pas écrire l'index pandas dans le fichier

# Sauvegarder les données enrichies
df.to_csv('../data/processed/benin_enrichi.csv', index=False)
# Même données mais avec les colonnes calculées

# Sauvegarder en Parquet (plus rapide et plus léger)
df.to_parquet('../data/processed/benin_enrichi.parquet', index=False)
# Format pour le dashboard Streamlit

print(" benin_clean.csv sauvegardé")
print(" benin_enrichi.csv sauvegardé")
print(" benin_enrichi.parquet sauvegardé")
print(f"\n Taille CSV : {os.path.getsize('../data/processed/benin_enrichi.csv') / 1024:.1f} KB")
print(f" Taille Parquet : {os.path.getsize('../data/processed/benin_enrichi.parquet') / 1024:.1f} KB")

 benin_clean.csv sauvegardé
 benin_enrichi.csv sauvegardé
 benin_enrichi.parquet sauvegardé

 Taille CSV : 3301.2 KB
 Taille Parquet : 587.8 KB


In [ ]:
### Cellule pour le rapport sur la qualité des données

print("=" * 55)  
# Afficher une ligne de séparation pour structurer le rapport

print(" RAPPORT QUALITÉ FINAL — PIPELINE J2")  
# Titre du rapport de qualité des données

print("=" * 55)  
# Deuxième ligne de séparation

print(f"Événements traités     : {len(df):,}")  
# Afficher le nombre total de lignes (événements), formaté avec séparateur de milliers

print(f"Période couverte       : {df['SQLDATE'].min().date()} → {df['SQLDATE'].max().date()}")  
# Afficher la période temporelle couverte par les données

print(f"Colonnes totales       : {df.shape[1]}")  
# Afficher le nombre total de colonnes

print(f"  dont originales      : 31")  
# Indiquer le nombre de colonnes d'origine (GDELT)

print(f"  dont enrichies       : {df.shape[1] - 31}")  
# Calculer et afficher le nombre de colonnes ajoutées après enrichissement

print(f"\n Valeurs manquantes clés :")  
# Section dédiée à l’analyse des valeurs manquantes

cols_a_verifier = ['GoldsteinScale', 'AvgTone', 'ActionGeo_Lat',
                   'Actor1Name', 'ActionGeo_ADM1Code']  
# Liste des colonnes critiques à analyser

for col in cols_a_verifier:  
    # Parcourir chaque colonne à vérifier
    if col in df.columns:  
        # Vérifier que la colonne existe dans le DataFrame
        n = df[col].isna().sum()  
        # Compter le nombre de valeurs manquantes
        pct = (n / len(df) * 100).round(1)  
        # Calculer le pourcentage de valeurs manquantes
        print(f"  {col:<25} : {n:,} ({pct}%)")  
        # Afficher les résultats alignés

print(f"\n Ton médiatique :")  
# Section sur la distribution du ton médiatique

print(df['ton_categorie'].value_counts().to_string())  
# Afficher le nombre d’occurrences par catégorie de ton

print(f"\n  Type d'événement (Goldstein) :")  
# Section sur la classification Goldstein

print(df['goldstein_categorie'].value_counts().to_string())  
# Afficher la distribution des catégories Goldstein

print(f"\n  QuadClass :")  
# Section sur les classes d’événements GDELT

print(df['quadclass_label'].value_counts().to_string())  
# Afficher la distribution des QuadClass

print(f"\n Zone géographique :")  
# Section sur la répartition géographique

print(df['zone_benin'].value_counts().to_string())  
# Afficher la distribution par zones (Nord, Centre, Sud)

print("\n" + "=" * 55)  
# Ligne de fin du rapport

 RAPPORT QUALITÉ FINAL — PIPELINE J2
Événements traités     : 10,722
Période couverte       : 2025-01-01 → 2025-12-31
Colonnes totales       : 41
  dont originales      : 31
  dont enrichies       : 10

 Valeurs manquantes clés :
  GoldsteinScale            : 0 (0.0%)
  AvgTone                   : 0 (0.0%)
  ActionGeo_Lat             : 0 (0.0%)
  Actor1Name                : 748 (7.0%)
  ActionGeo_ADM1Code        : 0 (0.0%)

 Ton médiatique :
ton_categorie
tres_negatif    3730
tres_positif    2091
neutre          1811
negatif         1668
positif         1422

  Type d'événement (Goldstein) :
goldstein_categorie
cooperatif          4964
conflictuel         1860
tres_cooperatif     1401
neutre              1297
tres_conflictuel    1200

  QuadClass :
quadclass_label
cooperation_verbale       6992
conflit_materiel          1558
conflit_verbal            1182
cooperation_materielle     990

 Zone géographique :
zone_benin
sud       10047
nord        523
centre      152

